# Cocogoat 2026 - Task 3: class-conditional latent GAN (z -> G -> z' -> Decoder -> D)

Stage A trains a beta-VAE on the Genshin set (1 h) and freezes it. Stage B trains a generator that works entirely in latent space (noise with the latent's shape -> displacement z' = z + G(z, class)), decoded by the frozen decoder and judged by two critics: an image critic on decoded fakes vs *decoded reals* and a latent critic on z' vs posterior samples (keeps z' on the decoder's manifold). Same CLIP pseudo-classes and metrics as Task 1, so the two are directly comparable. Budget: ~4 h. Outputs: `MyDrive/Cocogoat/runs/task3/`.

**How to run:** `Runtime -> Change runtime type -> G4 (RTX PRO 6000)`, set the budgets in the settings cell (5),
then `Runtime -> Run all`.
Keep this tab open (Colab Pro stops idle/closed sessions). Everything is written to
`MyDrive/Cocogoat/runs/...`; if Colab disconnects, just *Run all* again - training resumes from the last checkpoint.
You do not need to inspect results: metrics, sample grids and logs are saved to Drive for analysis.

In [ ]:
# 1) Mount Google Drive (outputs + datasets live there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Get / update the code (public repo, branch claude/cocogoat-revival-glfnmz)
import os, subprocess
REPO, BRANCH = "https://github.com/Martyn0324/Cocogoat.git", "claude/cocogoat-revival-glfnmz"
if not os.path.exists('/content/Cocogoat/.git'):
    subprocess.run(['git', 'clone', '-b', BRANCH, '--depth', '1', REPO, '/content/Cocogoat'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Cocogoat', 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Cocogoat', 'reset', '--hard', 'FETCH_HEAD'], check=True)
print(subprocess.run(['git', '-C', '/content/Cocogoat', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
os.chdir('/content/Cocogoat')

In [ ]:
# 3) Extra packages (torch/torchvision/transformers come with Colab)
!pip -q install -r /content/Cocogoat/requirements-colab.txt
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# Colab Secrets are only readable from the notebook kernel: export HF_TOKEN for the training subprocesses
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN exported')
except Exception as exc:
    print('No HF_TOKEN secret (only needed for gated Hugging Face datasets):', type(exc).__name__)
os.environ['PYTHONUNBUFFERED'] = '1'

In [ ]:
# 4) GPU smoke test (tiny models, a few steps, always from scratch)
!python -m cocogoat.tasks.task3_latent_gan --smoke --dataset genshin --run_dir /content/drive/MyDrive/Cocogoat/runs/task3_smoke --force

In [ ]:
# 5) Settings - edit freely. Training time budgets in hours: stage A = VAE, stage B = latent GAN.
#    Quick test: small budgets AND a separate folder (e.g. '.../runs/task3_test'): the finished VAE stage is not
#    extended later. (Raising GAN_HOURS on the same RUN_DIR does continue the GAN's training.)
VAE_HOURS, GAN_HOURS = 1.0, 3.0
RUN_DIR = '/content/drive/MyDrive/Cocogoat/runs/task3'

In [ ]:
# 6) VAE, then latent GAN (resumes automatically after a disconnect)
!python -m cocogoat.tasks.task3_latent_gan --dataset genshin --vae_hours {VAE_HOURS} --hours {GAN_HOURS} --run_dir {RUN_DIR}

In [ ]:
# 7) Results
from IPython.display import Image, Markdown, display
run = RUN_DIR
display(Markdown(open(f'{run}/gan/report.md').read()))
display(Image(f'{run}/vae/samples/recon_final.png'))
display(Image(f'{run}/gan/samples/final.png'))
display(Image(f'{run}/gan/samples/final_per_class.png'))